# Random sums & Wald's identity

_Probability (advanced)_

**Add up a random NUMBER of random pieces: the mean and variance of the total have two clean formulas.**

You know how to add up a fixed number of random numbers. A random sum makes the COUNT
        itself random: you draw a random whole number $N$, then add up $N$ independent pieces
        $X_1,\dots,X_N$.

       Two sources of randomness now stack. The pieces wobble (each $X_i$ is random), and the number of
        pieces wobbles (the count $N$ is random). The mean of the total is wonderfully simple — just average
        count times average piece. The variance has TWO parts, one from each source of wobble.

---

This notebook is a practice scaffold for the **Random sums & Wald's identity** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — NumPy

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# Random COUNT N ~ Poisson(lam):  E[N] = Var(N) = lam.
lam = 5.0
# i.i.d. PIECE X ~ Gamma(shape=k, scale=th), tuned to E[X]=3, Var(X)=4:
#   E[X] = k*th = 3,  Var(X) = k*th^2 = 4  ->  th = 4/3, k = 2.25
k, th = 2.25, 4.0 / 3.0
EX, VarX = k * th, k * th * th          # = 3.0, 4.0
EN, VarN = lam, lam                     # = 5.0, 5.0

trials = 2_000_000
N = rng.poisson(lam, size=trials)       # a random count per trial

# Draw one big pool of pieces, then slice it into the per-trial sums.
total_pieces = int(N.sum())
X = rng.gamma(k, th, size=total_pieces)
ends = np.cumsum(N)                      # end index of each trial's block
starts = ends - N                       # start index of each trial's block
csum = np.concatenate(([0.0], np.cumsum(X)))
Y = csum[ends] - csum[starts]           # Y_t = sum of that trial's pieces (0 if N=0)

EY_sim, VarY_sim = Y.mean(), Y.var()
EY_th = EN * EX                                  # Wald's identity
VarY_th = EN * VarX + EX**2 * VarN               # random-sum variance

print(f"E[X]={EX:.3f}  Var(X)={VarX:.3f}   E[N]={EN:.3f}  Var(N)={VarN:.3f}")
print(f"E[Y]   simulated = {EY_sim:8.4f}   theory E[N]E[X]               = {EY_th:8.4f}")
print(f"Var(Y) simulated = {VarY_sim:8.4f}   theory E[N]Var(X)+E[X]^2Var(N) = {VarY_th:8.4f}")
# E[X]=3.000  Var(X)=4.000   E[N]=5.000  Var(N)=5.000
# E[Y]   simulated =  14.9997   theory E[N]E[X]               =  15.0000
# Var(Y) simulated =  65.0144   theory E[N]Var(X)+E[X]^2Var(N) =  65.0000

## Visualize the data & results

_Do the random-sum formulas actually predict the simulated mean and variance? Compare simulated vs theoretical E[Y] and Var(Y) side by side._

In [ ]:
import numpy as np
rng = np.random.default_rng(7)

lam = 5.0                 # N ~ Poisson(5):  E[N] = Var(N) = 5
k, th = 2.25, 4.0 / 3.0   # X ~ Gamma -> E[X] = 3, Var(X) = 4
EX, VarX, EN, VarN = k * th, k * th * th, lam, lam

trials = 500_000
N = rng.poisson(lam, size=trials)
X = rng.gamma(k, th, size=int(N.sum()))
ends = np.cumsum(N); starts = ends - N
csum = np.concatenate(([0.0], np.cumsum(X)))
Y = csum[ends] - csum[starts]

EY_sim, VarY_sim = Y.mean(), Y.var()
EY_th = EN * EX                          # Wald: -> 15.0
VarY_th = EN * VarX + EX**2 * VarN       # random-sum variance: -> 65.0
# EY_sim -> 14.997, VarY_sim -> 64.932

labels = ['E[Y] sim', 'E[Y] theory', 'Var(Y) sim', 'Var(Y) theory']
values = [EY_sim, EY_th, VarY_sim, VarY_th]

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, values, color=['#4ea1ff', '#7ee787', '#ffb454', '#c89bff'])
ax.set_ylabel('value')
ax.set_title('Random sum: simulated mean/variance match the formulas')
for i, v in enumerate(values):
    ax.text(i, v + 0.5, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A store gets $N\sim\text{Poisson}(10)$ customers a day ($E[N]=10$, $\operatorname{Var}(N)=10$). Each customer spends an amount with mean $E[X]=20$ and variance $\operatorname{Var}(X)=50$, independent of the count. Find $E[Y]$ and $\operatorname{Var}(Y)$ for the total daily revenue $Y$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean by Wald: $E[Y] = E[N]\,E[X] = 10 \times 20$. — _The mean of a random sum is average count times average piece._
- Within-count term: $E[N]\,\operatorname{Var}(X) = 10 \times 50 = 500$. — _Spread from the piece sizes, scaled by the average number of pieces._
- Between-count term: $(E[X])^2\,\operatorname{Var}(N) = 20^2 \times 10 = 400 \times 10 = 4000$. — _Extra spread from the random count; the piece mean is squared._
- Add the two variance terms. — _Law of total variance: within plus between._

**Answer:** $E[Y] = 200$ and $\operatorname{Var}(Y) = 500 + 4000 = 4500$.

</details>

**Problem 2.** Why does $E[Y]=E[N]\,E[X]$ hold for any independent count distribution, while the second variance term $(E[X])^2\operatorname{Var}(N)$ can dominate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the tower rule: $E[Y]=E[E[Y\mid N]]=E[N\,E[X]]=E[N]\,E[X]$. — _Iterated expectation only needs $E[Y\mid N]=N\,E[X]$, which holds for any $N$._
- For the variance, the between-count term grows with $(E[X])^2$ and with $\operatorname{Var}(N)$. — _A large piece mean amplifies count uncertainty quadratically._

**Answer:** The mean only uses linearity through the tower rule, so it is distribution-free. But when the average piece $E[X]$ is large or the count is very uncertain (large $\operatorname{Var}(N)$), the $(E[X])^2\operatorname{Var}(N)$ term can swamp the within-count term — the uncertainty in HOW MANY pieces there are dominates the total spread.

</details>

**Problem 3.** The count is fixed at exactly $N=8$ (so $\operatorname{Var}(N)=0$). Each piece has $E[X]=2$, $\operatorname{Var}(X)=9$. What are $E[Y]$ and $\operatorname{Var}(Y)$, and which formula term disappears?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean: $E[Y]=E[N]\,E[X]=8\times 2=16$. — _Wald's identity with a degenerate (constant) count._
- Within-count term: $E[N]\,\operatorname{Var}(X)=8\times 9=72$. — _The only surviving variance term._
- Between-count term: $(E[X])^2\,\operatorname{Var}(N)=2^2\times 0=0$. — _A fixed count has zero variance._

**Answer:** $E[Y]=16$, $\operatorname{Var}(Y)=72$. The count-variance term $(E[X])^2\operatorname{Var}(N)$ vanishes, and we recover the ordinary fixed-sum result $\operatorname{Var}(Y)=n\,\operatorname{Var}(X)$.

</details>